In [ ]:
!pip install torch torchvision
!pip install pillow 
!pip install imagehash  
!pip install scipy 

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import time

# Set device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔍 Using device: {device}")

# Load DINOv2 ViT-g/14 model
start_time = time.time()
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14_reg').to(device)
model.eval()  # Set to evaluation mode
print(f"Model loaded in {time.time() - start_time:.2f} seconds")

# Define image preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize((518, 518)),  # Resize to match DINOv2 input size
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # Adjusted for DINOv2
])

def extract_features(image_path):
    """Extract deep features from an image using DINOv2."""
    try:
        print(f"Processing: {image_path}")
        start_time = time.time()
        
        img = Image.open(image_path).convert('RGB')  # Ensure RGB format
        img_tensor = preprocess(img).unsqueeze(0).to(device)  # Add batch dimension & move to device

        with torch.no_grad():
            features = model(img_tensor)  # Extract features

        # Convert tensor to numpy
        features = features.cpu().squeeze().numpy()
        print(f"Feature shape: {features.shape}")
        print(f"Processing time: {time.time() - start_time:.2f} seconds\n")
        return features

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

def compare_images_deep(img1_path, img2_path, threshold=90):
    """Compare two images using deep learning features from DINOv2."""
    start_time = time.time()
    
    features1 = extract_features(img1_path)
    features2 = extract_features(img2_path)

    if features1 is None or features2 is None:
        print("Feature extraction failed. Cannot compare images.")
        return None

    # Compute cosine similarity
    similarity = cosine_similarity(features1.reshape(1, -1), features2.reshape(1, -1))[0][0]

    # Convert similarity to a 0-100 scale
    similarity_score = round(similarity * 100, 2)

    result = {
        # "image1": img1_path,
        # "image2": img2_path,
        "similarity_score": similarity_score,
        "is_plagiarism": similarity_score > threshold
    }

    print(f"Total comparison time: {time.time() - start_time:.2f} seconds")
    print(f"Similarity Score: {similarity_score}/100\n")
    return result

# Example Usage
img1_path = "/workspace/Notebook/mec2_2.jpeg"
img2_path = "/workspace/Notebook/mec2.jpg"

result = compare_images_deep(img1_path, img2_path)
print(result)